# Agentic RAG demo (Google Colab)

**End-to-end demo for interviews:**
1. Clone repo and install dependencies
2. Download RAG datasets (FinQA + TAT-QA) via script — no manual upload
3. Build FinQA retriever index on GPU
4. Run RAG evaluation (FinQA + TAT-QA) and display results

**Before running:** Runtime → Change runtime type → **T4 GPU**. Set `ANTHROPIC_API_KEY` in Colab secrets or in a cell.

## 1. Clone repo and install dependencies

In [ ]:
REPO_URL = "https://github.com/your-org/ocr-agentic-rag.git"  # replace with your repo
REPO_DIR = "ocr-agentic-rag"
import os
if os.path.isdir(REPO_DIR):
    get_ipython().run_line_magic('cd', REPO_DIR)
    get_ipython().system('git pull')
else:
    get_ipython().system(f'git clone {REPO_URL} {REPO_DIR}')
get_ipython().run_line_magic('cd', REPO_DIR)

!pip install -q sentence-transformers faiss-cpu rank_bm25 llama-index-core anthropic langgraph pandas

## 2. Download RAG datasets (FinQA + TAT-QA)

Runs `scripts/download_rag_datasets.py` — no manual upload. To download only one dataset, run:
`!python scripts/download_rag_datasets.py --datasets finqa`

In [ ]:
!python scripts/download_rag_datasets.py --datasets finqa tatqa

## 3. Generate first_5_rows for FinQA (optional)

Same outcome as `data/generate_pq_first_5_rows.py` for FinQA; no extra deps.

In [ ]:
import json
from pathlib import Path

train_dir = Path("data/rag/FinQA/train")
train_qa_path = train_dir / "train_qa.json"
first_5_path = train_dir / "first_5_rows.json"
if train_qa_path.exists():
    with open(train_qa_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict) and "data" in data:
        data = data["data"]
    rows = data[:5] if isinstance(data, list) else []
    with open(first_5_path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)
    print(f"Wrote {first_5_path} ({len(rows)} rows).")
else:
    print("train_qa.json not found; run section 2 first.")

## 4. Build FinQA retriever index on GPU

In [ ]:
!python scripts/build_finqa_embeddings_colab.py --output data/rag/FinQA/train/finqa_retriever_index --batch_size 256

## 5. Run RAG evaluation (FinQA + TAT-QA)

Requires `ANTHROPIC_API_KEY` (Colab: Secrets or set in a cell). Uses small sample for a quick demo; increase `--max_split` / remove limits for full eval.

In [ ]:
# Set your API key if not in Colab Secrets
# import os
# os.environ["ANTHROPIC_API_KEY"] = "your-key"

!python eval_runner.py --category rag --max_split 1 --max_category 2 --debug --export_predictions_txt

## 6. Display results

In [ ]:
import json
from pathlib import Path

try:
    from IPython.display import Markdown, display
    import pandas as pd
    _use_display = True
except ImportError:
    _use_display = False

proof_dir = Path("data/proof/rag")
if not proof_dir.exists():
    print("No proof dir yet. Run section 5 first.")
else:
    if _use_display:
        display(Markdown("## RAG evaluation results"))
    for dataset_dir in sorted(proof_dir.iterdir()):
        if not dataset_dir.is_dir():
            continue
        name = dataset_dir.name.upper() if len(dataset_dir.name) <= 7 else dataset_dir.name
        weighted = dataset_dir / f"{dataset_dir.name}_weighted_avg.json"
        if weighted.exists():
            with open(weighted) as f:
                m = json.load(f)
            if _use_display:
                display(Markdown(f"### {name}"))
                rows = [(k, f"{v:.4f}" if isinstance(v, (int, float)) and v is not None else str(v)) for k, v in sorted(m.items())]
                display(pd.DataFrame(rows, columns=["Metric", "Value"]))
            else:
                print(f"### {name}")
                for k, v in sorted(m.items()):
                    print(f"  {k}: {v}")
        for split_dir in dataset_dir.iterdir():
            if not split_dir.is_dir():
                continue
            for f in split_dir.glob("*_per_sample_*.json"):
                with open(f) as fp:
                    samples = json.load(fp)
                if _use_display:
                    display(Markdown(f"**{dataset_dir.name}** ({split_dir.name}): **{len(samples)}** samples evaluated."))
                else:
                    print(f"{dataset_dir.name} ({split_dir.name}): {len(samples)} samples")
                break
            break
    summary = Path("data/proof/eval_summary.json")
    if summary.exists():
        with open(summary) as f:
            s = json.load(f)
        if _use_display:
            display(Markdown("### Eval summary"))
            if isinstance(s, dict):
                flat = [(k, json.dumps(v)[:80] if not isinstance(v, (int, float, str)) else v) for k, v in s.items()]
                display(pd.DataFrame(flat, columns=["Key", "Value"]))
            else:
                display(s)
        else:
            print("### Summary")
            print(json.dumps(s, indent=2))

## 7. (Optional) Download index bundle for local use

Run this to download the built FinQA index and use it locally so you can skip embedding there.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

src = Path("data/rag/FinQA/train/finqa_retriever_index")
if src.exists():
    zip_path = Path("finqa_retriever_index.zip")
    shutil.make_archive(zip_path.with_suffix(""), "zip", src.parent, src.name)
    files.download(str(zip_path))
    print("Downloaded. Unzip into data/rag/FinQA/train/ locally.")
else:
    print("Index not found; run section 4 first.")